# **Metode : KNN, Decision Tree, Metode Random Forest, Metode Bagging ,Metode Logistic Regression, Metode Naive Bayes, ADAboost, gradient boosting, catboost, xgboost. Dengan menggunakan SMOTE, Optimasi parameter dan Threshold Prediction**

In [ ]:
!pip install catboost
!pip install imbalanced-learn
!pip install openpyxl
!pip install xgboost
!pip install joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report
)

from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# **Load Data**

In [ ]:
file_path = "/content/DataSampelCekNutrisiMakananUAS.xlsx"

data = pd.read_excel(file_path)

print("Ukuran Data:", data.shape)
print(data.head())
print(data["Nutrisi"].value_counts())

Ukuran Data: (1650, 9)
                NamaMakanan  Energi(Kcal)  Protein(g)  Karbohidrat(g)  Fat(g)  \
0   Ikan sunu, asin, mentah         199.0        34.1             7.1     3.8   
1  Soto pekalongan, masakan          94.0         3.0             5.1     6.8   
2                 Marie duo          15.0         1.0            14.0     3.5   
3          Kaparende, sayur          38.0         2.4             2.6     2.0   
4       Kacang lebui / iris         346.0        16.5            66.6     1.5   

   Fiber(g)  Gula(g)  Sodium(g)  Nutrisi  
0       0.0      0.0      377.0        0  
1       0.3      0.0        0.0        0  
2       1.0      5.0       85.0        0  
3       1.5      0.0      170.0        0  
4      37.3      0.0       17.0        1  
Nutrisi
0    1196
1     454
Name: count, dtype: int64


# **PREPROCESSING DATA**

In [ ]:
target_col = "Nutrisi"

In [ ]:
if "NamaMakanan" in data.columns:
    data = data.drop(columns=["NamaMakanan"])

data[target_col] = pd.to_numeric(data[target_col], errors="coerce")
data = data.dropna(subset=[target_col])
data[target_col] = data[target_col].astype(int)

X = data.drop(columns=[target_col])
y = data[target_col]

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

numeric_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        )
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# **Fungsi Metode**

In [ ]:
def make_bagging():
    try:
        return BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=123),
            n_estimators=100,
            random_state=123
        )
    except TypeError:
        return BaggingClassifier(
            base_estimator=DecisionTreeClassifier(random_state=123),
            n_estimators=100,
            random_state=123
        )

In [ ]:
def make_adaboost():
    try:
        return AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=123),
            n_estimators=100,
            learning_rate=0.1,
            random_state=123
        )
    except TypeError:
        return AdaBoostClassifier(
            base_estimator=DecisionTreeClassifier(max_depth=1, random_state=123),
            n_estimators=100,
            learning_rate=0.1,
            random_state=123
        )

In [ ]:
def get_positive_probability(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    else:
        return model.predict(X_data)

In [ ]:
def hitung_metrik(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1_Score": f1_score(y_true, y_pred, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

In [ ]:
def cari_threshold_terbaik(model, X_valid, y_valid):
    prob = get_positive_probability(model, X_valid)

    best_threshold = 0.50
    best_score = -1

    for threshold in np.arange(0.10, 0.91, 0.01):
        y_pred = (prob >= threshold).astype(int)
        score = f1_score(y_valid, y_pred, zero_division=0)

        if score > best_score:
            best_score = score
            best_threshold = threshold

    return round(best_threshold, 2), best_score

# **Model Awal**

In [ ]:
models_awal = {
    "KNN": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=123
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=123
    ),

    "Bagging": make_bagging(),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=123
    ),

    "Naive Bayes": GaussianNB(),

    "AdaBoost": make_adaboost(),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=123
    ),

    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=4,
        verbose=0,
        random_state=123
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=123
    )
}

# **Parameter Optimasi Model**

In [ ]:
param_grids = {
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 15],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"]
    },

    "Decision Tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [2, 3, 5, 7, 10, None],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 5]
    },

    "Random Forest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [3, 5, 7, 10, None],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 5],
        "model__max_features": ["sqrt", "log2"]
    },

    "Bagging": {
        "model__n_estimators": [50, 100, 200],
        "model__max_samples": [0.6, 0.8, 1.0],
        "model__max_features": [0.6, 0.8, 1.0]
    },

    "Logistic Regression": {
        "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
        "model__solver": ["liblinear", "lbfgs"],
        "model__penalty": ["l2"]
    },

    "Naive Bayes": {
        "model__var_smoothing": np.logspace(-12, -6, 7)
    },

    "AdaBoost": {
        "model__n_estimators": [50, 100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]
    },

    "Gradient Boosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "model__max_depth": [2, 3, 4],
        "model__subsample": [0.7, 0.8, 1.0]
    },

    "CatBoost": {
        "model__iterations": [100, 200, 300],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__depth": [3, 4, 5, 6],
        "model__l2_leaf_reg": [1, 3, 5, 7]
    },

    "XGBoost": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4, 5],
        "model__subsample": [0.7, 0.8, 1.0],
        "model__colsample_bytree": [0.7, 0.8, 1.0]
    }
}

# **Split Data**

In [ ]:
rasio_data = {
    "80_20": 0.20,
    "90_10": 0.10
}

hasil_evaluasi = []
hasil_parameter = []
hasil_threshold = []

In [ ]:
for nama_rasio, test_size in rasio_data.items():

    print("\n" + "="*70)
    print(f"PEMBAGIAN DATA {nama_rasio}")
    print("="*70)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=123
    )

    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.20,
        stratify=y_train_full,
        random_state=123
    )

    print("Jumlah Data Training :", len(X_train_full))
    print("Jumlah Data Testing  :", len(X_test))
    print("Jumlah Data Validasi :", len(X_valid))

    best_models = {}

    for nama_model, estimator in models_awal.items():

        print("\n" + "-"*60)
        print(f"Model: {nama_model}")
        print("-"*60)

        pipe_awal = Pipeline(steps=[
            ("prep", preprocessor),
            ("model", estimator)
        ])

        pipe_awal.fit(X_train, y_train)
        y_pred_awal = pipe_awal.predict(X_test)

        metrik_awal = hitung_metrik(y_test, y_pred_awal)
        hasil_evaluasi.append({
            "Rasio_Data": nama_rasio,
            "Model": nama_model,
            "Skenario": "Model Awal",
            "Threshold": 0.50,
            **metrik_awal
        })

        pipe_smote = ImbPipeline(steps=[
            ("prep", preprocessor),
            ("smote", SMOTE(random_state=123)),
            ("model", estimator)
        ])
        pipe_smote.fit(X_train, y_train)
        y_pred_smote = pipe_smote.predict(X_test)

        metrik_smote = hitung_metrik(y_test, y_pred_smote)
        hasil_evaluasi.append({
            "Rasio_Data": nama_rasio,
            "Model": nama_model,
            "Skenario": "SMOTE",
            "Threshold": 0.50,
            **metrik_smote
        })

        pipe_tuning = ImbPipeline(steps=[
            ("prep", preprocessor),
            ("smote", SMOTE(random_state=123)),
            ("model", estimator)
        ])

        jumlah_kombinasi = 1
        for values in param_grids[nama_model].values():
            jumlah_kombinasi *= len(values)

        n_iter = min(30, jumlah_kombinasi)

        search = RandomizedSearchCV(
            estimator=pipe_tuning,
            param_distributions=param_grids[nama_model],
            n_iter=n_iter,
            scoring="f1",
            cv=cv,
            n_jobs=-1,
            random_state=123,
            refit=True
        )

        search.fit(X_train, y_train)

        best_model = search.best_estimator_
        best_models[nama_model] = best_model

        y_pred_opt = best_model.predict(X_test)

        metrik_opt = hitung_metrik(y_test, y_pred_opt)
        hasil_evaluasi.append({
            "Rasio_Data": nama_rasio,
            "Model": nama_model,
            "Skenario": "SMOTE + Optimasi Parameter",
            "Threshold": 0.50,
            **metrik_opt
        })

        params_bersih = {
            key.replace("model__", ""): value
            for key, value in search.best_params_.items()
        }

        hasil_parameter.append({
            "Rasio_Data": nama_rasio,
            "Model": nama_model,
            "Parameter_Optimal": params_bersih,
            "Best_CV_F1": search.best_score_
        })

        print("Parameter Optimal:")
        print(params_bersih)

        best_threshold, best_valid_f1 = cari_threshold_terbaik(
            best_model,
            X_valid,
            y_valid
        )

        prob_test = get_positive_probability(best_model, X_test)
        y_pred_threshold = (prob_test >= best_threshold).astype(int)

        metrik_threshold = hitung_metrik(y_test, y_pred_threshold)

        hasil_evaluasi.append({
            "Rasio_Data": nama_rasio,
            "Model": nama_model,
            "Skenario": "SMOTE + Optimasi Parameter + Threshold Prediction",
            "Threshold": best_threshold,
            **metrik_threshold
        })

        hasil_threshold.append({
            "Rasio_Data": nama_rasio,
            "Model": nama_model,
            "Best_Threshold": best_threshold,
            "Best_Valid_F1": best_valid_f1
        })

        print("Threshold Optimal:", best_threshold)



PEMBAGIAN DATA 80_20
Jumlah Data Training : 1320
Jumlah Data Testing  : 330
Jumlah Data Validasi : 264

------------------------------------------------------------
Model: KNN
------------------------------------------------------------
Parameter Optimal:
{'weights': 'distance', 'n_neighbors': 9, 'metric': 'manhattan'}
Threshold Optimal: 0.42

------------------------------------------------------------
Model: Decision Tree
------------------------------------------------------------
Parameter Optimal:
{'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 7, 'criterion': 'gini'}
Threshold Optimal: 0.26

------------------------------------------------------------
Model: Random Forest
------------------------------------------------------------
Parameter Optimal:
{'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}
Threshold Optimal: 0.46

------------------------------------------------------------
Model: Bagging
------

In [ ]:
evaluasi_df = pd.DataFrame(hasil_evaluasi)
parameter_df = pd.DataFrame(hasil_parameter)
threshold_df = pd.DataFrame(hasil_threshold)

ranking_df = evaluasi_df.sort_values(
    by=["F1_Score", "Balanced_Accuracy", "Accuracy"],
    ascending=False
).reset_index(drop=True)

print("\nHASIL EVALUASI SEMUA MODEL")
print(evaluasi_df)

print("\nPARAMETER OPTIMAL")
print(parameter_df)

print("\nTHRESHOLD OPTIMAL")
print(threshold_df)

print("\nRANKING MODEL TERBAIK")
print(ranking_df.head(20))


HASIL EVALUASI SEMUA MODEL
    Rasio_Data                Model  \
0        80_20                  KNN   
1        80_20        Decision Tree   
2        80_20        Random Forest   
3        80_20              Bagging   
4        80_20  Logistic Regression   
..         ...                  ...   
169      90_10             CatBoost   
170      90_10              XGBoost   
171      90_10              XGBoost   
172      90_10              XGBoost   
173      90_10              XGBoost   

                                              Skenario  Threshold  Accuracy  \
0                                           Model Awal       0.50  0.975758   
1                                           Model Awal       0.50  0.966667   
2                                           Model Awal       0.50  0.969697   
3                                           Model Awal       0.50  0.969697   
4                                           Model Awal       0.50  0.984848   
..                           

In [ ]:
best_row = ranking_df.iloc[0]

print("\nModel Terbaik:")
print(best_row)


Model Terbaik:
Rasio_Data                         80_20
Model                Logistic Regression
Skenario                           SMOTE
Threshold                            0.5
Accuracy                        0.987879
Precision                       0.978022
Recall                          0.978022
F1_Score                        0.978022
Balanced_Accuracy               0.984827
TN                                   237
FP                                     2
FN                                     2
TP                                    89
Name: 0, dtype: object


# **Matrix Confussion**

In [ ]:
confusion_matrix_df = evaluasi_df[
    [
        "Rasio_Data",
        "Model",
        "Skenario",
        "Threshold",
        "TN",
        "FP",
        "FN",
        "TP"
    ]
].copy()

confusion_matrix_df["Confusion_Matrix"] = confusion_matrix_df.apply(
    lambda row: f"[[{row['TN']}, {row['FP']}], [{row['FN']}, {row['TP']}]]",
    axis=1
)


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [ ]:
confusion_matrix_df = evaluasi_df[
    [
        "Rasio_Data",
        "Model",
        "Skenario",
        "Threshold",
        "TN",
        "FP",
        "FN",
        "TP"
    ]
].copy()

confusion_matrix_df["Confusion_Matrix"] = confusion_matrix_df.apply(
    lambda row: f"""
Prediksi
          0      1
Aktual 0  {row['TN']}     {row['FP']}
Aktual 1  {row['FN']}     {row['TP']}
""",
    axis=1
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

for rasio in confusion_matrix_df["Rasio_Data"].unique():

    print("\n" + "="*90)
    print(f"CONFUSION MATRIX DATA SPLIT {rasio}")
    print("="*90)

    data_rasio = confusion_matrix_df[
        confusion_matrix_df["Rasio_Data"] == rasio
    ]

    for model in data_rasio["Model"].unique():

        print("\n" + "-"*90)
        print(f"MODEL : {model}")
        print("-"*90)

        data_model = data_rasio[
            data_rasio["Model"] == model
        ]

        for _, row in data_model.iterrows():

            print(f"\nSkenario  : {row['Skenario']}")
            print(f"Threshold : {row['Threshold']}")
            print(row["Confusion_Matrix"])


CONFUSION MATRIX DATA SPLIT 80_20

------------------------------------------------------------------------------------------
MODEL : KNN
------------------------------------------------------------------------------------------

Skenario  : Model Awal
Threshold : 0.5

Prediksi
          0      1
Aktual 0  238     1
Aktual 1  7     84


Skenario  : Model Awal
Threshold : 0.5

Prediksi
          0      1
Aktual 0  238     1
Aktual 1  7     84


Skenario  : SMOTE
Threshold : 0.5

Prediksi
          0      1
Aktual 0  234     5
Aktual 1  2     89


Skenario  : SMOTE + Optimasi Parameter
Threshold : 0.5

Prediksi
          0      1
Aktual 0  236     3
Aktual 1  2     89


Skenario  : SMOTE + Optimasi Parameter + Threshold Prediction
Threshold : 0.42

Prediksi
          0      1
Aktual 0  235     4
Aktual 1  2     89


Skenario  : Model Awal
Threshold : 0.5

Prediksi
          0      1
Aktual 0  238     1
Aktual 1  7     84


Skenario  : SMOTE
Threshold : 0.5

Prediksi
          0      1
A

# **Perbandingan dan Pemilihan Model Terbaik**

In [ ]:
urutan_model = [
    "CatBoost",
    "XGBoost",
    "KNN",
    "Gradient Boosting",
    "AdaBoost",
    "Decision Tree",
    "Random Forest",
    "Bagging",
    "Logistic Regression",
    "Naive Bayes"
]

mapping_skenario = {
    "Model Awal": "Model Awal",
    "SMOTE": "SMOTE",
    "SMOTE + Optimasi Parameter": "Optimasi Parameter",
    "SMOTE + Optimasi Parameter + Threshold Prediction": "Threshold Prediction"
}

evaluasi_tabel = evaluasi_df.copy()
evaluasi_tabel["Skenario_Tabel"] = evaluasi_tabel["Skenario"].map(mapping_skenario)

metrik_evaluasi = [
    "Accuracy",
    "Balanced_Accuracy",
    "Precision",
    "Recall",
    "F1_Score"
]

tabel_ringkasan = {}

for metrik in metrik_evaluasi:

    tabel = evaluasi_tabel.pivot_table(
        index="Model",
        columns="Skenario_Tabel",
        values=metrik,
        aggfunc="mean"
    )

    tabel = tabel.reindex(urutan_model)

    tabel = tabel[
        [
            "Model Awal",
            "SMOTE",
            "Optimasi Parameter",
            "Threshold Prediction"
        ]
    ]

    tabel = tabel.round(4)
    tabel_ringkasan[metrik] = tabel

    print("\n" + "="*80)
    print(f"TABEL {metrik.upper()}")
    print("="*80)
    print(tabel)


TABEL ACCURACY
Skenario_Tabel       Model Awal   SMOTE  Optimasi Parameter  Threshold Prediction
Model                                                                            
CatBoost                 0.9682  0.9742              0.9717                0.9717
XGBoost                  0.9733  0.9737              0.9798                0.9758
KNN                      0.9727  0.9773              0.9803                0.9758
Gradient Boosting        0.9621  0.9697              0.9682                0.9667
AdaBoost                 0.9394  0.9576              0.9697                0.9636
Decision Tree            0.9621  0.9576              0.9530                0.9591
Random Forest            0.9667  0.9727              0.9652                0.9636
Bagging                  0.9667  0.9636              0.9697                0.9682
Logistic Regression      0.9833  0.9818              0.9833                0.9788
Naive Bayes              0.9576  0.9561              0.9561                0.9576


In [ ]:
for rasio in evaluasi_tabel["Rasio_Data"].unique():

    data_rasio = evaluasi_tabel[
        evaluasi_tabel["Rasio_Data"] == rasio
    ]

    print("\n" + "#"*90)
    print(f"RINGKASAN EVALUASI UNTUK SPLIT {rasio}")
    print("#"*90)

    for metrik in metrik_evaluasi:

        tabel = data_rasio.pivot_table(
            index="Model",
            columns="Skenario_Tabel",
            values=metrik,
            aggfunc="mean"
        )

        tabel = tabel.reindex(urutan_model)

        tabel = tabel[
            [
                "Model Awal",
                "SMOTE",
                "Optimasi Parameter",
                "Threshold Prediction"
            ]
        ]

        tabel = tabel.round(4)

        print("\n" + "="*80)
        print(f"{metrik.upper()} - SPLIT {rasio}")
        print("="*80)
        print(tabel)


##########################################################################################
RINGKASAN EVALUASI UNTUK SPLIT 80_20
##########################################################################################

ACCURACY - SPLIT 80_20
Skenario_Tabel       Model Awal   SMOTE  Optimasi Parameter  Threshold Prediction
Model                                                                            
CatBoost                 0.9727  0.9788              0.9727                0.9727
XGBoost                  0.9758  0.9758              0.9818                0.9758
KNN                      0.9758  0.9788              0.9848                0.9818
Gradient Boosting        0.9667  0.9697              0.9727                0.9697
AdaBoost                 0.9455  0.9455              0.9697                0.9697
Decision Tree            0.9667  0.9576              0.9485                0.9606
Random Forest            0.9697  0.9758              0.9727                0.9758
Bagging           

In [ ]:
urutan_model = [
    "CatBoost",
    "XGBoost",
    "KNN",
    "Gradient Boosting",
    "AdaBoost",
    "Decision Tree",
    "Random Forest",
    "Bagging",
    "Logistic Regression",
    "Naive Bayes"
]

nama_skenario = {
    "Model Awal": "Model Awal",
    "SMOTE": "SMOTE",
    "SMOTE + Optimasi Parameter": "SMOTE + Optimasi Parameter",
    "SMOTE + Optimasi Parameter + Threshold Prediction": "SMOTE + Optimasi + Threshold"
}

evaluasi_tabel = evaluasi_df.copy()
evaluasi_tabel["Skenario"] = evaluasi_tabel["Skenario"].replace(nama_skenario)

metrik_evaluasi = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1_Score",
    "Balanced_Accuracy"
]

In [ ]:
tabel_per_metrik = {}

for rasio in evaluasi_tabel["Rasio_Data"].unique():

    data_rasio = evaluasi_tabel[
        evaluasi_tabel["Rasio_Data"] == rasio
    ]

    for metrik in metrik_evaluasi:

        tabel = data_rasio.pivot_table(
            index="Model",
            columns="Skenario",
            values=metrik,
            aggfunc="first"
        )

        tabel = tabel.reindex(urutan_model)
        tabel = tabel.round(4)

        tabel_per_metrik[f"{rasio}_{metrik}"] = tabel

        print("\n" + "="*100)
        print(f"TABEL {metrik.upper()} - SPLIT {rasio}")
        print("="*100)
        print(tabel)



TABEL ACCURACY - SPLIT 80_20
Skenario             Model Awal   SMOTE  SMOTE + Optimasi + Threshold  SMOTE + Optimasi Parameter
Model                                                                                            
CatBoost                 0.9727  0.9788                        0.9727                      0.9727
XGBoost                  0.9758  0.9758                        0.9758                      0.9818
KNN                      0.9758  0.9788                        0.9818                      0.9848
Gradient Boosting        0.9667  0.9697                        0.9697                      0.9727
AdaBoost                 0.9455  0.9455                        0.9697                      0.9697
Decision Tree            0.9667  0.9576                        0.9606                      0.9485
Random Forest            0.9697  0.9758                        0.9758                      0.9727
Bagging                  0.9697  0.9636                        0.9727                   

In [ ]:
evaluasi_tabel["Ringkasan_Metrik"] = evaluasi_tabel.apply(
    lambda row:
    f"Acc={row['Accuracy']:.4f}; "
    f"BalAcc={row['Balanced_Accuracy']:.4f}; "
    f"Prec={row['Precision']:.4f}; "
    f"Rec={row['Recall']:.4f}; "
    f"F1={row['F1_Score']:.4f}",
    axis=1
)

tabel_ringkasan = {}

for rasio in evaluasi_tabel["Rasio_Data"].unique():

    data_rasio = evaluasi_tabel[
        evaluasi_tabel["Rasio_Data"] == rasio
    ]

    tabel = data_rasio.pivot_table(
        index="Model",
        columns="Skenario",
        values="Ringkasan_Metrik",
        aggfunc="first"
    )

    tabel = tabel.reindex(urutan_model)

    tabel_ringkasan[rasio] = tabel

    print("\n" + "="*120)
    print(f"TABEL RINGKASAN EVALUASI MODEL - SPLIT {rasio}")
    print("="*120)
    print(tabel)



TABEL RINGKASAN EVALUASI MODEL - SPLIT 80_20
Skenario                                                    Model Awal                                              SMOTE                       SMOTE + Optimasi + Threshold  \
Model                                                                                                                                                                          
CatBoost             Acc=0.9727; BalAcc=0.9574; Prec=0.9767; Rec=0....  Acc=0.9788; BalAcc=0.9649; Prec=0.9884; Rec=0....  Acc=0.9727; BalAcc=0.9710; Prec=0.9362; Rec=0....   
XGBoost              Acc=0.9758; BalAcc=0.9628; Prec=0.9770; Rec=0....  Acc=0.9758; BalAcc=0.9663; Prec=0.9663; Rec=0....  Acc=0.9758; BalAcc=0.9663; Prec=0.9663; Rec=0....   
KNN                  Acc=0.9758; BalAcc=0.9594; Prec=0.9882; Rec=0....  Acc=0.9788; BalAcc=0.9786; Prec=0.9468; Rec=0....  Acc=0.9818; BalAcc=0.9806; Prec=0.9570; Rec=0....   
Gradient Boosting    Acc=0.9667; BalAcc=0.9464; Prec=0.9762; Rec=0....  Ac

# **Pengecekan Over Fitting**

In [ ]:
hasil_overfitting = []

def evaluasi_overfitting(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1_Score": f1_score(y_true, y_pred, zero_division=0)
    }

In [ ]:
for rasio in rasio_data.keys():

    print("\n" + "="*90)
    print(f"PENGECEKAN OVERFITTING SPLIT {rasio}")
    print("="*90)

    for nama_model in best_models.keys():

        model = best_models[nama_model]

        # Prediksi data training
        y_train_pred = model.predict(X_train_full)

        # Prediksi data testing
        y_test_pred = model.predict(X_test)

        train_score = evaluasi_overfitting(y_train_full, y_train_pred)
        test_score = evaluasi_overfitting(y_test, y_test_pred)

        gap_accuracy = train_score["Accuracy"] - test_score["Accuracy"]
        gap_balanced_accuracy = train_score["Balanced_Accuracy"] - test_score["Balanced_Accuracy"]
        gap_f1 = train_score["F1_Score"] - test_score["F1_Score"]

        if gap_f1 > 0.10:
            status = "Overfitting"
        elif gap_f1 < -0.05:
            status = "Underfitting / Tidak Stabil"
        else:
            status = "Tidak Overfitting"

        hasil_overfitting.append({
            "Rasio_Data": rasio,
            "Model": nama_model,
            "Train_Accuracy": train_score["Accuracy"],
            "Test_Accuracy": test_score["Accuracy"],
            "Gap_Accuracy": gap_accuracy,
            "Train_Balanced_Accuracy": train_score["Balanced_Accuracy"],
            "Test_Balanced_Accuracy": test_score["Balanced_Accuracy"],
            "Gap_Balanced_Accuracy": gap_balanced_accuracy,
            "Train_F1": train_score["F1_Score"],
            "Test_F1": test_score["F1_Score"],
            "Gap_F1": gap_f1,
            "Status": status
        })


PENGECEKAN OVERFITTING SPLIT 80_20

PENGECEKAN OVERFITTING SPLIT 90_10


In [ ]:
overfitting_df = pd.DataFrame(hasil_overfitting)

In [ ]:
overfitting_df = overfitting_df.round({
    "Train_Accuracy": 4,
    "Test_Accuracy": 4,
    "Gap_Accuracy": 4,
    "Train_Balanced_Accuracy": 4,
    "Test_Balanced_Accuracy": 4,
    "Gap_Balanced_Accuracy": 4,
    "Train_F1": 4,
    "Test_F1": 4,
    "Gap_F1": 4
})

In [ ]:
print("\nHASIL PENGECEKAN OVERFITTING")
print(overfitting_df)


HASIL PENGECEKAN OVERFITTING
   Rasio_Data                Model  Train_Accuracy  Test_Accuracy  Gap_Accuracy  Train_Balanced_Accuracy  Test_Balanced_Accuracy  Gap_Balanced_Accuracy  Train_F1  Test_F1  Gap_F1             Status
0       80_20                  KNN          0.9933         0.9758        0.0175                   0.9923                  0.9694                 0.0229    0.9878   0.9556  0.0322  Tidak Overfitting
1       80_20        Decision Tree          0.9926         0.9576        0.0350                   0.9888                  0.9431                 0.0458    0.9865   0.9213  0.0651  Tidak Overfitting
2       80_20        Random Forest          0.9886         0.9576        0.0310                   0.9868                  0.9500                 0.0368    0.9793   0.9231  0.0562  Tidak Overfitting
3       80_20              Bagging          0.9939         0.9636        0.0303                   0.9935                  0.9542                 0.0394    0.9890   0.9333  0.0557